# Initialization

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import StringType
from pyspark.sql.window import Window

# Check data integration

In [0]:
TABLE_CONFIG = [
  "bronze.erp_loc_a101",
  "bronze.crm_cust_info",
  "bronze.erp_cust_az12"
]

for table in TABLE_CONFIG:
  spark.read.table(table).limit(50).display()

# Read the table

In [0]:
df = spark.read.table("bronze.erp_loc_a101")

# Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

# Clean the CID column

In [0]:
df = df.withColumn("CID", regexp_replace(col("CID"), "-", ""))

# Check standardization on CNTRY column

In [0]:
df.select("CNTRY").distinct().display()

# Normalize CNTRY column

In [0]:
df = df.withColumn("CNTRY", when(col("CNTRY").isin("US", "USA"), "United States")
              .when(col("CNTRY") == "DE", "Germany")
              .when((col("CNTRY").isNull()) | (col("CNTRY") == ""), "N/A")
              .otherwise(col("CNTRY"))
              )

# Sanity Check

In [0]:
df.display()